<a href="https://colab.research.google.com/github/algulawani/RM-Research-Final-Task/blob/main/face_alignment_train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install face-alignment

  Using cached face_alignment-1.4.1-py2.py3-none-any.whl.metadata (7.4 kB)
Using cached face_alignment-1.4.1-py2.py3-none-any.whl (30 kB)


In [4]:
"""
This script extracts facial landmarks and confidence scores from
all images in the RAF-DB training set using the face-alignment library.

To avoid running out of RAM on Google Colab, images are processed
in batches of 500. For each batch:
1. The face-alignment neural network is loaded into GPU memory
2. Each image in the batch is loaded, resized to 224x224, and
   passed through the detector to get 68 landmarks and scores
3. Results are appended to a running dictionary
4. The detector is deleted and GPU memory is cleared
5. The dictionary is saved to disk as a checkpoint

The dictionary is saved as a .pth file after every batch, so if
Colab disconnects, progress is not lost. On restart, the script
checks which images have already been processed and skips them.

The final output is a .pth file containing a dictionary where:
- Keys are image paths like "1/train_05178_aligned.jpg"
- Values are dictionaries with:
  - "landmarks": numpy array of shape (68, 2)
  - "scores": numpy array of shape (68,)
"""

import os
import gc
import numpy as np
import torch
import face_alignment
from PIL import Image

train_path = "/content/drive/MyDrive/OADN/RAF-DB/2/DATASET/train"
output_dir = "/content/drive/MyDrive/OADN/Preprocessed Training Data"
output_path = os.path.join(output_dir, "train_landmarks_confidencescores.pth")
os.makedirs(output_dir, exist_ok=True)

BATCH_SIZE = 500

# Collect all image paths
all_images = []
class_folders = sorted(os.listdir(train_path))
for class_folder in class_folders:
    class_path = os.path.join(train_path, class_folder)
    if not os.path.isdir(class_path):
        continue
    for img_name in sorted(os.listdir(class_path)):
        all_images.append(f"{class_folder}/{img_name}")

# Load existing progress if the script was interrupted previously
if os.path.exists(output_path):
    landmarks_dict = torch.load(output_path, weights_only=False)
    print(f"Loaded existing checkpoint with {len(landmarks_dict)} images")
else:
    landmarks_dict = {}

# Filter out already processed images
remaining_images = [img for img in all_images if img not in landmarks_dict]
print(f"Total images: {len(all_images)}")
print(f"Already processed: {len(landmarks_dict)}")
print(f"Remaining: {len(remaining_images)}")

failed_images = []

# Process in batches
for batch_start in range(0, len(remaining_images), BATCH_SIZE):
    batch = remaining_images[batch_start:batch_start + BATCH_SIZE]
    batch_num = (batch_start // BATCH_SIZE) + 1
    total_batches = (len(remaining_images) + BATCH_SIZE - 1) // BATCH_SIZE
    print(f"\nBatch {batch_num}/{total_batches}: processing {len(batch)} images")

    # Load the detector for this batch
    fa = face_alignment.FaceAlignment(
        face_alignment.LandmarksType.TWO_D,
        device='cuda'
    )

    for img_rel_path in batch:
        img_path = os.path.join(train_path, img_rel_path)

        image = Image.open(img_path).convert('RGB')
        image_resized = image.resize((224, 224), Image.BILINEAR)
        image_np = np.array(image_resized)

        try:
            landmarks, scores, _ = fa.get_landmarks_from_image(
                image_np,
                return_landmark_score=True
            )
        except Exception as e:
            failed_images.append(img_rel_path)
            continue

        if landmarks is None or len(landmarks) == 0:
            failed_images.append(img_rel_path)
            continue

        landmarks_dict[img_rel_path] = {
            "landmarks": landmarks[0],
            "scores": scores[0]
        }

    # Delete detector and free GPU memory
    del fa
    gc.collect()
    torch.cuda.empty_cache()

    # Save checkpoint after each batch
    torch.save(landmarks_dict, output_path)
    print(f"Checkpoint saved: {len(landmarks_dict)} images processed so far")

print(f"\nDone. Successfully processed: {len(landmarks_dict)} images")
print(f"Failed/skipped: {len(failed_images)} images")

Total images: 12271
Already processed: 0
Remaining: 12271

Batch 1/25: processing 500 images
Downloading: "https://www.adrianbulat.com/downloads/python-fan/s3fd-619a316812.pth" to /root/.cache/torch/hub/checkpoints/s3fd-619a316812.pth


100%|██████████| 85.7M/85.7M [00:07<00:00, 12.7MB/s]
Downloading: "https://www.adrianbulat.com/downloads/python-fan/2DFAN4-cd938726ad.zip" to /root/.cache/torch/hub/checkpoints/2DFAN4-cd938726ad.zip
100%|██████████| 91.9M/91.9M [00:06<00:00, 15.8MB/s]


Checkpoint saved: 500 images processed so far

Batch 2/25: processing 500 images


/usr/local/lib/python3.12/dist-packages/face_alignment/api.py:147: UserWarning: No faces were detected.
  warnings.warn("No faces were detected.")


Checkpoint saved: 995 images processed so far

Batch 3/25: processing 500 images
Checkpoint saved: 1493 images processed so far

Batch 4/25: processing 500 images
Checkpoint saved: 1990 images processed so far

Batch 5/25: processing 500 images
Checkpoint saved: 2485 images processed so far

Batch 6/25: processing 500 images
Checkpoint saved: 2983 images processed so far

Batch 7/25: processing 500 images
Checkpoint saved: 3483 images processed so far

Batch 8/25: processing 500 images
Checkpoint saved: 3982 images processed so far

Batch 9/25: processing 500 images
Checkpoint saved: 4481 images processed so far

Batch 10/25: processing 500 images
Checkpoint saved: 4981 images processed so far

Batch 11/25: processing 500 images
Checkpoint saved: 5481 images processed so far

Batch 12/25: processing 500 images
Checkpoint saved: 5980 images processed so far

Batch 13/25: processing 500 images
Checkpoint saved: 6480 images processed so far

Batch 14/25: processing 500 images
Checkpoint s